In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

from model.actor_critic import FrameObservationEncoderNet, EncoderNet
from dataset import get_dataloader

from tqdm import trange
import tqdm

In [2]:
def cosine_loss(x, y):
    return 1. - F.cosine_similarity(x, y, dim=-1).mean()

def mse_loss(x, y):
    return F.mse_loss(x, y, reduction="mean")

In [3]:
class Alignment(nn.Module):
    def __init__(self, state_encoder, frame_encoder, state_feature_layer=-1):
        super().__init__()

        self.state_feature_layer = state_feature_layer

        self.state_encoder = state_encoder

        self.frame_encoder = frame_encoder

        self.state_encoder.eval()

        for param in self.state_encoder.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def encode_states(self, vectors):
        state_features = self.state_encoder.get_features(vectors)

        return state_features[self.state_feature_layer]

    def encode_frames(self, frames):
        frame_features = self.frame_encoder(frames, False, False)

        return frame_features
    
    def forward(self, frames, states):
        frame_features = self.encode_frames(frames)
        state_features = self.encode_states(states)

        return frame_features, state_features

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 20
encoder_weight, actor_weight, critic_wegit = torch.load("state_model.pth", weights_only=True)

state_encoder = EncoderNet(6+6+3+4+3+4, [256, 256, 256]).to(device)
frame_encoder = FrameObservationEncoderNet(6, state_encoder.dim).to(device)

state_encoder.load_state_dict(encoder_weight)

model = Alignment(state_encoder, frame_encoder, -1).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer=optimizer, T_max=epochs)
dataloader = get_dataloader()
size = len(dataloader.dataset)
print(size)

160000


In [5]:
for _ in trange(epochs, desc="Epochs"):
    running_loss = 0.0
    running_cosine_loss = 0.0
    for _, (vectors, frames) in enumerate(dataloader):
        vectors = vectors.to(device)
        frames = frames.to(device)
        
        frame_features, vector_features = model(frames, vectors)
        mse_losss_value = mse_loss(frame_features, vector_features)
        cosine_loss_value = cosine_loss(frame_features, vector_features)
        loss = 0.5 * mse_losss_value + 0.5 * cosine_loss_value
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * vectors.size(0)
        running_cosine_loss += cosine_loss_value.item() * vectors.size(0)
    
    scheduler.step()
    train_loss = running_loss / size
    train_cosine_loss = running_cosine_loss / size
    tqdm.tqdm.write(f"Train Loss: {train_loss:.4f}, Cosine Loss: {train_cosine_loss:.4f}")

torch.save([model.frame_encoder.state_dict(), actor_weight, critic_wegit], "frame_model.pth")

Epochs:   5%|▌         | 1/20 [01:56<36:47, 116.20s/it]

Train Loss: 0.2372, Cosine Loss: 0.1431


Epochs:  10%|█         | 2/20 [03:52<34:52, 116.23s/it]

Train Loss: 0.0894, Cosine Loss: 0.0523


Epochs:  15%|█▌        | 3/20 [05:47<32:44, 115.57s/it]

...

Train Loss: 0.0133, Cosine Loss: 0.0073


Epochs:  95%|█████████▌| 19/20 [36:26<01:55, 115.18s/it]

Train Loss: 0.0117, Cosine Loss: 0.0064


Epochs: 100%|██████████| 20/20 [38:22<00:00, 115.13s/it]

Train Loss: 0.0108, Cosine Loss: 0.0058
